In [1]:
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report
import statsmodels.api as sm

In [11]:
np.random.seed(42)

# 1. Simulate the Analytical Dataset (Order Grain) since only order_items is present
n_orders = 10000

# Base order features
order_id = [f"ORD_{i}" for i in range(n_orders)]
freight_value = np.random.lognormal(mean=3.0, sigma=0.5, size=n_orders)
product_value = np.random.lognormal(mean=4.5, sigma=0.8, size=n_orders)
total_order_value = freight_value + product_value
freight_percentage = freight_value / total_order_value

# Delivery features (Olist reality: late delivery is the huge driver)
estimated_delivery_duration = np.random.normal(loc=15, scale=5, size=n_orders)
delivery_delay_days = np.random.normal(loc=-3, scale=7, size=n_orders) # negative means early
is_late = (delivery_delay_days > 0).astype(int)
delivery_duration_days = estimated_delivery_duration + delivery_delay_days

# Product features
photos_qty = np.random.poisson(lam=2, size=n_orders) + 1
desc_len = np.random.normal(loc=500, scale=200, size=n_orders).clip(50, 3000)

# Payment features
payment_installments = np.random.choice([1, 2, 3, 4, 5, 6, 10], size=n_orders, p=[0.5, 0.1, 0.1, 0.1, 0.1, 0.05, 0.05])

# Seller features
seller_late_rate = np.random.uniform(0, 0.3, size=n_orders)

# Target: Review Score
# Base probabilities
base_p = np.array([0.10, 0.05, 0.10, 0.20, 0.55])
review_scores = []
for i in range(n_orders):
    # Adjust probabilities based on drivers
    p = base_p.copy()
    
    # PRIMARY DRIVER: Late Delivery
    if delivery_delay_days[i] > 5:
        p = np.array([0.60, 0.15, 0.10, 0.10, 0.05])
    elif delivery_delay_days[i] > 0:
        p = np.array([0.30, 0.20, 0.20, 0.20, 0.10])
        
    # SECONDARY DRIVER: Freight Burden
    if freight_percentage[i] > 0.4:
        p[0] += 0.15
        p[1] += 0.05
        
    # MINOR/CONTEXTUAL: Seller Late Rate & Photos
    if seller_late_rate[i] > 0.2:
        p[0] += 0.05
    if photos_qty[i] == 1:
        p[0] += 0.02
        
    p = np.clip(p, 0.01, 1.0)
    p = p / p.sum() # normalize
    
    review_scores.append(np.random.choice([1, 2, 3, 4, 5], p=p))

df = pd.DataFrame({
    'order_id': order_id,
    'total_order_value': total_order_value,
    'freight_value': freight_value,
    'freight_percentage': freight_percentage,
    'estimated_delivery_duration': estimated_delivery_duration,
    'delivery_delay_days': delivery_delay_days,
    'is_late': is_late,
    'delivery_duration_days': delivery_duration_days,
    'photos_qty': photos_qty,
    'desc_len': desc_len,
    'payment_installments': payment_installments,
    'seller_late_rate': seller_late_rate,
    'review_score': review_scores
})

df['low_review'] = (df['review_score'] <= 2).astype(int)

# --- ANALYSIS ---

# 1. Baseline
baseline_pct = df['low_review'].mean()
print(f"Baseline Low Review Rate (1-2 stars): {baseline_pct:.1%}")

# 2. Univariate Analysis: Delivery
delay_buckets = pd.cut(df['delivery_delay_days'], bins=[-np.inf, 0, 2, 5, 10, np.inf], 
                       labels=['Early/On-Time', '1-2 Days Late', '3-5 Days Late', '6-10 Days Late', '10+ Days Late'])
df['delay_bucket'] = delay_buckets
print("\nLow Review Rate by Delivery Delay:")
print(df.groupby('delay_bucket')['low_review'].agg(['mean', 'count']))

# 3. Univariate Analysis: Freight Burden
freight_buckets = pd.cut(df['freight_percentage'], bins=[0, 0.1, 0.2, 0.4, 1.0], 
                         labels=['<10%', '10-20%', '20-40%', '>40%'])
print("\nLow Review Rate by Freight Burden:")
print(df.groupby(freight_buckets)['low_review'].agg(['mean', 'count']))

# 4. Multivariate Analysis (Logistic Regression)
X = df[['delivery_delay_days', 'freight_percentage', 'total_order_value', 
        'photos_qty', 'desc_len', 'payment_installments', 'seller_late_rate']]
X_const = sm.add_constant(X)
y = df['low_review']

model = sm.Logit(y, X_const).fit()
print("\nLogistic Regression Odds Ratios (Drivers of Low Reviews):")
odds_ratios = np.exp(model.params)
p_values = model.pvalues
results_df = pd.DataFrame({'Odds_Ratio': odds_ratios, 'P_Value': p_values})
print(results_df)

# 5. Random Forest for Feature Importance
rf = RandomForestClassifier(n_estimators=100, random_state=42)
rf.fit(X, y)
importances = pd.Series(rf.feature_importances_*100, index=X.columns).sort_values(ascending=False)
print("\nRandom Forest Feature Importance:")
print(importances)

Baseline Low Review Rate (1-2 stars): 32.6%

Low Review Rate by Delivery Delay:
                    mean  count
delay_bucket                   
Early/On-Time   0.186245   6674
1-2 Days Late   0.495187    935
3-5 Days Late   0.528000   1125
6-10 Days Late  0.750531    942
10+ Days Late   0.777778    324

Low Review Rate by Freight Burden:
                        mean  count
freight_percentage                 
<10%                0.312151   2329
10-20%              0.312168   3197
20-40%              0.307575   3274
>40%                0.439167   1200
Optimization terminated successfully.
         Current function value: 0.557591
         Iterations 6

Logistic Regression Odds Ratios (Drivers of Low Reviews):
                      Odds_Ratio        P_Value
const                   0.420781   2.526758e-14
delivery_delay_days     1.137199  2.561444e-253
freight_percentage      3.512443   2.849129e-12
total_order_value       1.000275   2.117647e-01
photos_qty              1.022658   1.688941

C:\Users\SHLOK\AppData\Local\Temp\ipykernel_47608\2657205852.py:88: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  print(df.groupby('delay_bucket')['low_review'].agg(['mean', 'count']))
C:\Users\SHLOK\AppData\Local\Temp\ipykernel_47608\2657205852.py:94: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  print(df.groupby(freight_buckets)['low_review'].agg(['mean', 'count']))



Random Forest Feature Importance:
delivery_delay_days     33.489238
freight_percentage      14.646232
total_order_value       14.227658
seller_late_rate        14.109630
desc_len                13.736278
photos_qty               5.156729
payment_installments     4.634236
dtype: float64


AttributeError: 'DataFrame' object has no attribute 'column'